# Animal Multi-Day Behavior Workflow

This notebook collects behavioral sessions for a single animal, converts Unity JSON logs to HDF5 files, and assembles multi-day trial summaries with rolling accuracy metrics. Adjust the parameters below and run the notebook top-to-bottom.

In [5]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "behavioral_analysis").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repository root starting from {start}"
    )


repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repository root: {repo_root}")
print(f"Added to sys.path: {src_path}")

Repository root: /groups/spruston/home/moharb/DELTA_Behavior/notebooks


In [ ]:
# Parameters
ANIMAL_ID = "BM35"  # Update with the animal identifier of interest
SEARCH_ROOT = Path("/nearline/spruston/Boaz/Mesoscope") / ANIMAL_ID  # Directory to search for JSON logs (recursive)
OUTPUT_HDF5_DIR = repo_root / "outputs" / "hdf5" / ANIMAL_ID
ROLLING_WINDOW = 20  # Number of trials in the running accuracy window
OVERWRITE_EXISTING = False  # Set True to regenerate HDF5 files even if they exist
SKIP_FAILED_SESSIONS = True  # Continue when a JSON log is missing required events

/groups/spruston/home/moharb/DELTA_Behavior/notebooks


In [3]:
import pandas as pd
from behavioral_analysis.analysis import collect_json_logs

json_paths = collect_json_logs(ANIMAL_ID, SEARCH_ROOT)
if not json_paths:
    raise FileNotFoundError(f"No JSON logs for {ANIMAL_ID} under {SEARCH_ROOT}")

json_summary = pd.DataFrame({
    "json_path": [path.as_posix() for path in json_paths]
})
json_summary

In [ ]:
from behavioral_analysis.analysis import convert_json_sessions

conversion_result = convert_json_sessions(
    json_paths,
    animal_id=ANIMAL_ID,
    output_dir=OUTPUT_HDF5_DIR,
    overwrite=OVERWRITE_EXISTING,
    verbose=True,
    skip_failures=SKIP_FAILED_SESSIONS,
    return_failures=True,
)

if isinstance(conversion_result, tuple):
    conversions, conversion_failures = conversion_result
else:  # Backward compatibility
    conversions = conversion_result
    conversion_failures = []

if not conversions:
    raise RuntimeError("No sessions were successfully converted; adjust SEARCH_ROOT or ensure logs contain cue data.")

conversion_summary = pd.DataFrame({
    "session_order": range(1, len(conversions) + 1),
    "session_label": [c.session_label for c in conversions],
    "session_date": [c.session_date for c in conversions],
    "session_number": [c.session_number for c in conversions],
    "json_path": [c.json_path.as_posix() for c in conversions],
    "hdf5_path": [c.hdf5_path.as_posix() for c in conversions],
}).sort_values("session_date").reset_index(drop=True)
conversion_summary

In [ ]:
if conversion_failures:
    failure_summary = pd.DataFrame([
        {"json_path": failure.json_path.as_posix(), "error": failure.error}
        for failure in conversion_failures
    ])
    display(failure_summary)
else:
    print("No conversion failures.")

In [ ]:
from behavioral_analysis.analysis import prepare_multi_day_trials

multi_day = prepare_multi_day_trials(conversions, rolling_window=ROLLING_WINDOW)
multi_day.session_summary

In [ ]:
daily_accuracy = (
    multi_day.trials.groupby("session_date")
    ["correct"]
    .mean()
    .mul(100.0)
    .reset_index(name="accuracy_pct")
)
daily_accuracy

In [ ]:
multi_day.trials.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(
    multi_day.trials["trial_index_global"],
    multi_day.trials["rolling_accuracy_pct"],
    color="navy",
    linewidth=2,
    label=f"Rolling accuracy ({ROLLING_WINDOW} trials)",
)
ax.axhline(50, color="gray", linestyle=":", linewidth=1, alpha=0.7)
ax.set_xlabel("Global trial index")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)

for transition in multi_day.day_transitions[1:]:
    ax.axvline(
        transition.first_trial_index - 0.5,
        linestyle="--",
        color="gray",
        linewidth=1,
        alpha=0.6,
    )

ymin, ymax = ax.get_ylim()
label_y = ymax - (ymax - ymin) * 0.05
for idx, transition in enumerate(multi_day.day_transitions, start=1):
    ax.text(
        transition.first_trial_index,
        label_y,
        transition.session_date.isoformat(),
        rotation=90,
        va="top",
        ha="left",
        fontsize=9,
        color="dimgray",
    )

ax.set_title(f"{ANIMAL_ID} multi-day running accuracy", pad=20)
ax.legend(loc="lower right")
fig.tight_layout()
fig